In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/us-birth/annotated/checkpoints/post_cell_2.pickle

trying: ['plt']
me:  0
trying: ['sns']
me:  0
trying: ['birth_data']


me:  3
trying: ['pd']
me:  0
trying: ['np']
me:  0
trying: ['time']
me:  0
trying: ['factor']
me:  3
trying: ['orig_output']
me:  6
Declaring variable plt
Declaring variable sns
Declaring variable pd
Declaring variable np
Declaring variable time
Declaring variable birth_data
Declaring variable factor
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 3 ###

# Chain fillna and astype into a single pipeline to cut down on intermediate GPU operations
birth_data = birth_data.fillna(0).astype({ 'day': 'int64' })
birth_data.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 7773500 entries, 0 to 15546
Data columns (total 5 columns):
 #   Column  Dtype
---  ------  -----
 0   year    int64
 1   month   int64
 2   day     int64
 3   gender  object
 4   births  int64
dtypes: int64(4), object(1)
memory usage: 333.6+ MB
CPU times: user 4.71 ms, sys: 24.2 ms, total: 29 ms
Wall time: 28.9 ms


In [5]:
%Checkpoint /home/dias-benchmarks/new_notebooks/us-birth/rewritten/q20_rewrite/checkpoints/post_cell_3_try_0.pickle

migration speed (bps): 784472216.9385382
---------------------------
variables to migrate:
sns 72
pd 72
np 72
orig_output 112
time 72
birth_data 48
plt 72
factor 28
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/new_notebooks/us-birth/rewritten/q20_rewrite/checkpoints/post_cell_3_try_0.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['birth_data']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['birth_data']
Intermediate variables ['factor']
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['birth_data']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['birth_data']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/new_notebooks/us-birth/opt_cell_exec_info_3_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
birth_data_opt = birth_data
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/us-birth/annotated/checkpoints/post_cell_3.pickle
assert compare_df(birth_data_opt, birth_data)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
